In [1]:
from utils_shiprocket import prepare_data
from utils_shiprocket import evaluate_binary_classifier

train_df,val_df,test_df = prepare_data(train_examples = None)

Resolving data files:   0%|          | 0/83 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/83 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/82 [00:00<?, ?it/s]

In [9]:
from tqdm.auto import tqdm

In [2]:
import torch
import torch.nn as nn
from transformers import AutoFeatureExtractor, Wav2Vec2Model

class Wav2Vec2GRUClassifier(nn.Module):
    def __init__(self, pretrained_model_name="facebook/wav2vec2-base", gru_hidden_size=128, gru_num_layers=1, freeze_entire_wav2vec=True, dropout=0.2, cnn_channels=256, cnn_kernel_size=5):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(pretrained_model_name)
        if freeze_entire_wav2vec:
            for param in self.wav2vec2.parameters():
                param.requires_grad = False
        hidden_size = self.wav2vec2.config.hidden_size
        self.cnn = nn.Sequential(
            nn.Conv1d(hidden_size, cnn_channels, kernel_size=cnn_kernel_size, padding=cnn_kernel_size // 2),
            # nn.BatchNorm1d(cnn_channels),
            nn.GELU(),
            nn.Conv1d(cnn_channels, cnn_channels, kernel_size=cnn_kernel_size, padding=cnn_kernel_size // 2),
            # nn.BatchNorm1d(cnn_channels),
            nn.GELU(),
        )
        self.gru = nn.GRU(input_size=cnn_channels, hidden_size=gru_hidden_size, num_layers=gru_num_layers, batch_first=True, bidirectional=False)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(gru_hidden_size, 1)

    def forward(self, input_values, attention_mask=None):
        if not next(self.wav2vec2.parameters()).requires_grad:
            with torch.no_grad():
                outputs = self.wav2vec2(input_values, attention_mask=attention_mask)
        else:
            outputs = self.wav2vec2(input_values, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state
        cnn_out = self.cnn(hidden_states.transpose(1, 2)).transpose(1, 2)
        if attention_mask is not None:
            feat_lengths = torch.clamp(self.wav2vec2._get_feat_extract_output_lengths(attention_mask.sum(-1).long()), min=1)
            packed = nn.utils.rnn.pack_padded_sequence(cnn_out, feat_lengths.cpu(), batch_first=True, enforce_sorted=False)
            _, h_n = self.gru(packed)
        else:
            _, h_n = self.gru(cnn_out)
        pooled = self.dropout(h_n[-1])
        return self.classifier(pooled)

feature_extractor = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base")
model = Wav2Vec2GRUClassifier()

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from utils_shiprocket import give_item_y_and_sr

class Wav2VecDataset(Dataset):
    def __init__(self, df, feature_extractor):
        self.df = df
        self.feature_extractor = feature_extractor

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, idx):
        # Ensure give_item_y_and_sr returns a 1D numpy array of audio values
        aud_value, sr = give_item_y_and_sr(self.df[idx], sr=16000)
        y = self.df[idx]['endpoint_bool']

        inputs = self.feature_extractor(
            aud_value,
            sampling_rate=16000,
            return_tensors="pt"
        )

        audio_tensor = inputs.input_values.squeeze(0)
        return audio_tensor, torch.tensor(y, dtype=torch.float32)

def collate_fn(batch):
    X, y = zip(*batch)
    lengths = torch.tensor([x.shape[0] for x in X], dtype=torch.long)
    X = pad_sequence(X, batch_first=True, padding_value=0.0)
    y = torch.stack(y).unsqueeze(1)  # (batch, 1) to match model output
    return X, lengths, y

In [7]:
def lengths_to_attention_mask(lengths, max_len):
    batch_size = lengths.size(0)
    mask = torch.arange(max_len).expand(batch_size, max_len) < lengths.unsqueeze(1)
    return mask.long()

def evaluate_full(loader, model, loss_fn):
    model.eval()
    total_loss, y_true, y_pred = 0.0, [], []
    with torch.no_grad():
        for X, lengths, y in tqdm(loader):
            attention_mask = lengths_to_attention_mask(lengths, X.size(1))
            logits = model(X, attention_mask=attention_mask)
            loss = loss_fn(logits, y)
            total_loss += loss.item() * X.size(0)
            pred = (torch.sigmoid(logits) >= 0.5).long()
            y_pred.extend(pred.cpu().tolist())
            y_true.extend(y.long().cpu().tolist())
    metrics = evaluate_binary_classifier(y_true, y_pred)
    metrics["loss"] = total_loss / len(loader.dataset)
    model.train()
    return metrics


In [5]:
# Usage
train_dataset = Wav2VecDataset(train_df,feature_extractor)
val_dataset = Wav2VecDataset(val_df,feature_extractor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

model = Wav2Vec2GRUClassifier()
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4
)

EPOCHS = 5
PRINT_EVERY = 1
val_iter = iter(val_loader)
print(len(train_loader))

try:
    for epoch in range(EPOCHS):
        model.train()
        for batch_idx, (X, lengths, y) in enumerate(train_loader):
            optimizer.zero_grad()
            attention_mask = lengths_to_attention_mask(lengths, X.size(1))
            logits = model(X, attention_mask=attention_mask)
            train_loss = loss_fn(logits, y)
            train_loss.backward()
            optimizer.step()

            try:
                X_val, val_lengths, y_val = next(val_iter)
            except StopIteration:
                val_iter = iter(val_loader)
                X_val, val_lengths, y_val = next(val_iter)

            model.eval()
            with torch.no_grad():
                val_attention_mask = lengths_to_attention_mask(val_lengths, X_val.size(1))
                val_logits = model(X_val, attention_mask=val_attention_mask)
                val_loss = loss_fn(val_logits, y_val)
                val_pred = (torch.sigmoid(val_logits) >= 0.5).long()
            model.train()

            metrics = evaluate_binary_classifier(y_val.long().cpu().tolist(), val_pred.cpu().tolist())

            if batch_idx % PRINT_EVERY == 0:
                print(f"Epoch {epoch:02d} | Batch {batch_idx:04d} | train_loss={train_loss.item():.4f} | val_loss={val_loss.item():.4f} | accuracy={metrics['accuracy']:.4f} | f1={metrics['f1_score']:.4f}")

        full_val_metrics = evaluate_full(val_loader, model, loss_fn)
        print(f"=== End of Epoch {epoch:02d} | full_val_loss={full_val_metrics['loss']:.4f} | accuracy={full_val_metrics['accuracy']:.4f} | f1={full_val_metrics['f1_score']:.4f} ===")

except KeyboardInterrupt:
    print(f"\nStopped at Epoch {epoch}, Batch {batch_idx}")
    torch.save({"model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(), "epoch": epoch, "batch_idx": batch_idx}, "interrupted_checkpoint_wav2vec.pt")
    print("Checkpoint saved.")

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


3891
Epoch 00 | Batch 0000 | train_loss=0.6930 | val_loss=0.6940 | accuracy=0.5000 | f1=0.0000
Epoch 00 | Batch 0001 | train_loss=0.6818 | val_loss=0.6771 | accuracy=0.7500 | f1=0.0000
Epoch 00 | Batch 0002 | train_loss=0.6902 | val_loss=0.6930 | accuracy=0.4375 | f1=0.0000
Epoch 00 | Batch 0003 | train_loss=0.7187 | val_loss=0.6696 | accuracy=0.7500 | f1=0.0000
Epoch 00 | Batch 0004 | train_loss=0.7184 | val_loss=0.6957 | accuracy=0.4375 | f1=0.1818
Epoch 00 | Batch 0005 | train_loss=0.6739 | val_loss=0.6943 | accuracy=0.5625 | f1=0.3636
Epoch 00 | Batch 0006 | train_loss=0.6911 | val_loss=0.6980 | accuracy=0.5000 | f1=0.3333
Epoch 00 | Batch 0007 | train_loss=0.7096 | val_loss=0.7036 | accuracy=0.4375 | f1=0.0000
Epoch 00 | Batch 0008 | train_loss=0.6969 | val_loss=0.6904 | accuracy=0.5625 | f1=0.2222
Epoch 00 | Batch 0009 | train_loss=0.6951 | val_loss=0.6923 | accuracy=0.4375 | f1=0.1818
Epoch 00 | Batch 0010 | train_loss=0.6855 | val_loss=0.6834 | accuracy=0.6250 | f1=0.5714
Epoch

In [ ]:
X_small, lengths_small, y_small = next(iter(train_loader))
for i in range(200):
    optimizer.zero_grad()
    mask = lengths_to_attention_mask(lengths_small, X_small.size(1))
    logits = model(X_small, attention_mask=mask)
    loss = loss_fn(logits, y_small)
    loss.backward()

    if i == 0:
        print("classifier grad mean:", model.classifier.weight.grad.abs().mean().item())
        print("gru grad mean:", model.gru.weight_ih_l0.grad.abs().mean().item())
        print("cnn grad mean:", model.cnn[0].weight.grad.abs().mean().item())
        print("logits:", logits.detach().flatten())
        print("labels:", y_small.flatten())
        print("X_small shape:", X_small.shape)
        print("lengths_small:", lengths_small)

    optimizer.step()
    print(i, loss.item())

In [ ]:
outputs = model.wav2vec2(X_small, attention_mask=mask)
hidden_states = outputs.last_hidden_state
cnn_out = model.cnn(hidden_states.transpose(1, 2)).transpose(1, 2)
feat_lengths = model.wav2vec2._get_feat_extract_output_lengths(mask.sum(-1))

print("cnn_out shape:", cnn_out.shape)
print("feat_lengths:", feat_lengths)
print("feat_lengths max:", feat_lengths.max().item())

In [ ]:
print("cnn_out min/max/mean:", cnn_out.min().item(), cnn_out.max().item(), cnn_out.mean().item())
print("fraction of zeros:", (cnn_out == 0).float().mean().item())

In [ ]:
full_val_metrics = evaluate_full(val_loader, model, loss_fn)
print(f"=== End of Epoch {epoch:02d} | full_val_loss={full_val_metrics['loss']:.4f} | accuracy={full_val_metrics['accuracy']:.4f} | f1={full_val_metrics['f1_score']:.4f} ===")

  0%|          | 0/487 [00:00<?, ?it/s]